In [ ]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os

# Accès à Google drive
SCOPES = ['https://www.googleapis.com/auth/drive.readonly'] # SCOPES = variable qui contient la liste des permissions. En l'occurence, lecture seule "readonly"

# 1/ Preuve d'identité
def connect_to_drive():
    creds = None
    # est ce que j'ai déjà des ID ?
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES) 
        #Credentials = classe founie par biblio google-auth. Gère les informations d’authentification nécessaires pour accéder aux API Google.
        #from_authorized_user_file méthode de classe qui recrée les ID d’un .json qui contient: access_token, refresh_token, client ID et client secret, scopes autorisés
    # Sinon, je me connecte manuellement à Google
    else:
        flow = InstalledAppFlow.from_client_secrets_file('credentials3.json', SCOPES)
        # InstalledAppFlow.from_client_secrets_file(...) Charge le fichier credentials.json fourni par Google Cloud Console
        creds = flow.run_local_server(port=0) 
        # flow.run_local_server(port=0) = Lance un mini serveur local sur ton ordi, 
        # Ouvre navigateur pour te faire connecter à Google, 
        # autorises l’accès à ton Drive via l’écran de consentement Google
        # ✅ Une fois validé : Google redirige l’utilisateur vers localhost avec un code d’autorisation 
        # Le script récupère ce code, puis échange ce code contre : un access_token, un refresh_token
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
        # Sauvegarde les jetons dans un fichier local token.json
        # Ce fichier est ensuite réutilisé automatiquement à chaque exécution suivante, tant que le token est valide.

# 2/ Connexion à l'API et utlisation dans le reste du script
    service = build('drive', 'v3', credentials=creds)
    # buil=fonction dans laquelle j'indique mes 3 params : 'drive' = service Google Drive, 'v3'=version de l'API, 'credentials'=objet OAuth 2.0
    # Tu ne définis pas les identifiants directement dans build, mais tu lui passes un objet Credentials déjà préparé avec : token.json ou credentials.json via l’authentification
    return service

# 3/ Accès au service
# Exemple : lister les fichiers PDF
def list_pdfs(service):
    results = service.files().list( #Requête à Google Drive via l'API, filtré par
        q="mimeType='application/pdf'", #uniquement des fichiers pdf
        pageSize=10, #10 résultats max
        fields="files(id, name)").execute() #on ne prend que l'ID et le nom
    
    items = results.get('files', []) #pour chaque fichier, tu affiches
    for item in items:
        print(f"{item['name']} ({item['id']})") #son nom, puis son ID

service = connect_to_drive()
list_pdfs(service)


Histoire toutes les leçons CM1 A4.pdf (1lad6YB8RPnyPWZ1n2Swl-uQNAAurlpe7)
Histoire_CM1.pdf (1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe)
template.pdf (1NqOs1QXrQb-bm7geitZw1RJh_MaAjp7l)
quizz_template.pdf (1xyYL20KiQw9UT9FNtvX3rDEC7bykD1L2)
exemple_quiz_revolution_francaise.pdf (1C6E5wN97lw-YJW3i_AoN70AZ7CQ2mPed)
histoire-CM1-emanuel.pdf (1cGcGr-YYfIxUzYwmHHcxcp7kL-fuppqA)
exemple_quiz_napoleon.pdf (1x3Uj8nxTzL8rkdA3F7Twp375NDwMbJ-s)
programmes_cycle-3_2023.pdf (14Ygx3xAUPhsQrluW3x119y4iJwLksFxi)
ensel714_annexe2_1312887.pdf (1I0_YSSrIZdMX3DOMBXlV0K1ibslCK5fG)
RA16_C3_HIGE_CM1_Th3_temps_Revolution_et_Empire_619871.pdf (1XTEndg-nbdU0E59QZD9ZwkHBU9FGXnBV)


In [5]:
#Variante affiche + 10 pdf
def list_all_pdfs(service):
    page_token = None
    while True:
        response = service.files().list(
            q="mimeType='application/pdf'",
            fields="nextPageToken, files(id, name)",
            pageToken=page_token
        ).execute()
        for file in response.get('files', []):
            print(f"{file['name']} ({file['id']})")
        page_token = response.get('nextPageToken', None)
        if page_token is None:
            break
service = connect_to_drive()
list_pdfs(service)

Histoire toutes les leçons CM1 A4.pdf (1lad6YB8RPnyPWZ1n2Swl-uQNAAurlpe7)
Histoire_CM1.pdf (1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe)
template.pdf (1NqOs1QXrQb-bm7geitZw1RJh_MaAjp7l)
quizz_template.pdf (1xyYL20KiQw9UT9FNtvX3rDEC7bykD1L2)
exemple_quiz_revolution_francaise.pdf (1C6E5wN97lw-YJW3i_AoN70AZ7CQ2mPed)
histoire-CM1-emanuel.pdf (1cGcGr-YYfIxUzYwmHHcxcp7kL-fuppqA)
exemple_quiz_napoleon.pdf (1x3Uj8nxTzL8rkdA3F7Twp375NDwMbJ-s)
programmes_cycle-3_2023.pdf (14Ygx3xAUPhsQrluW3x119y4iJwLksFxi)
ensel714_annexe2_1312887.pdf (1I0_YSSrIZdMX3DOMBXlV0K1ibslCK5fG)
RA16_C3_HIGE_CM1_Th3_temps_Revolution_et_Empire_619871.pdf (1XTEndg-nbdU0E59QZD9ZwkHBU9FGXnBV)


In [6]:
# lire le contenu
from googleapiclient.http import MediaIoBaseDownload
import io
from PyPDF2 import PdfReader

def lire_pdf_drive(service, file_id):
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()

    buffer.seek(0)
    reader = PdfReader(buffer)
    for i, page in enumerate(reader.pages):
        print(f"--- Page {i+1} ---\n{page.extract_text()}")



In [ ]:
#test si connexion OK
import io
from googleapiclient.http import MediaIoBaseDownload
from PyPDF2 import PdfReader

def lire_pdf_depuis_drive(service, file_id):
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    buffer.seek(0)
    reader = PdfReader(buffer)
    texte = ""
    for page in reader.pages:
        texte += page.extract_text() + "\n"
    return texte

# Utilisation
if __name__ == "__main__":
    service = connect_to_drive()
    file_id = "1lad6YB8RPnyPWZ1n2Swl-uQNAAurlpe7"  # Remplace par l'ID réel
    texte_pdf = lire_pdf_depuis_drive(service, file_id)
    print("=== Contenu du PDF ===\n")
    print(texte_pdf)


=== Contenu du PDF ===

HISTOIRE
CM1
SOMMAIREHISTOIRE1Clovis, roi des Francs2Charlemagne3La vie au Moyen-Age4La fin du Moyen-Age5Les grandes découvertes6La Renaissance7Les réformes religieuses8La Monarchie absolue9Le siècle des lumières
AuVèmesièclede406à500ontlieulesGrandesInvasions:despeuples,lesBarbaresvenantdetoutel’EuropeontenvahilaGaule.Maisseulunseuldecespeuples,lesFrancss’y’installèrent.Ilsfontainsidisparaîtrel’EmpireRomain(476).LeurroyaumecomprendlaGauleduNordetlaBelgique,puiss’étendàtoutelaGaule.LesFrancsetlesGallo-Romainssepartagentlespostesimportantsetsemariententreeux.Cettebonneententepermetàlaroyautéfranquedes’installerdurablementenGaulealorsquelesautresroyaumesbarbaresdisparaissentl’unaprèsl’autre.
1⃣Que s’est-il passé au Vèmesiècle ?___________________________________
2⃣Quel peuple s’installe en Gaule ?__________________________________
3⃣Quels étaient les deux peuples vivant en Gaule ? 1) _________________________________2) ________________________________Les  grandes 